# Webtoon panel detector — Colab / Kaggle GPU training

Runtime: **GPU** (Colab: Runtime → Change runtime type → T4). ~1–2 h.

Produces `best.onnx` (~12 MB). Download it and drop it into
`pipeline/models/manga-panel-yolo/manga_panel_detector_fp32_1024.onnx`.

You need a YOLO dataset folder (`images/{train,val}`, `labels/{train,val}`,
`data.yaml`) from `build_dataset.py` — upload it as a zip, or mount Drive.

In [ ]:
!pip -q install ultralytics onnx onnxslim
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# --- get the dataset ---
# Option A: upload webtoon-yolo.zip via the Files panel, then:
!unzip -q -o webtoon-yolo.zip -d /content/data
DATA = '/content/data/webtoon-yolo/data.yaml'

# Option B (Colab + Drive):
# from google.colab import drive; drive.mount('/content/drive')
# DATA = '/content/drive/MyDrive/webtoon-yolo/data.yaml'

import yaml; print(yaml.safe_load(open(DATA)))

In [ ]:
from ultralytics import YOLO

# start from the general YOLOv8n (or 'yolo11n.pt'). Fine-tuning, not scratch.
model = YOLO('yolov8n.pt')

model.train(
    data=DATA,
    epochs=120,
    imgsz=1024,              # webtoon pages are tall — keep resolution up
    batch=16,
    rect=False,
    mosaic=0.5, mixup=0.0, copy_paste=0.0,
    degrees=0.0, shear=0.0, perspective=0.0,   # comics aren't rotated
    fliplr=0.0,               # reading order / speech-tail direction matters
    hsv_h=0.0, hsv_s=0.3, hsv_v=0.3,
    patience=25,
    project='/content/runs', name='webtoon',
)

In [ ]:
# --- eval + export ---
best = '/content/runs/webtoon/weights/best.pt'
m = YOLO(best)
print(m.val(data=DATA, imgsz=1024).box.map, 'mAP50-95')

onnx_path = m.export(format='onnx', imgsz=1024, opset=13, simplify=True, nms=True)
print('exported:', onnx_path)

from google.colab import files            # Colab: download
files.download(onnx_path)

## Deploy

```bash
cp best.onnx pipeline/models/manga-panel-yolo/manga_panel_detector_fp32_1024.onnx
python -m pytest tests/test_image_slicing.py -q
```

If the exported output shape differs from the current `(1, N, 6)` `[x1,y1,x2,y2,
score,cls]`, adjust `_yolo_detect_page` in `master_pipeline.py` (it already
handles a class dimension; `nms=True` in the export keeps it close).

With a `bubble` class trained in, you can also drop the interim RT-DETR wiring
(`RECAP_TEXT_DET_SPLIT`) since one pass now yields both.